# 5 類純災害分類（Kaggle，Run All）

拿掉 MEDIC 的「Other / No Disaster」（not_disaster + other_disaster），只分 5 類真實災害（地震/淹水/火災/颱風/土石流），量測「5 類能到多高」，判斷階層式 cascade 是否值得做。

訓練兩個模型對照：從零自訓 CNN 與 ImageNet 預訓練 ResNet18 全層微調，皆在同一 test split 評估。

## 使用前
1. 開 **GPU + Internet**。
2. （可選）Add-ons → Secrets 設 `HF_TOKEN` 加速下載。
3. **Run All**。重點看 classification_report 的 **macro avg F1** 與各類 recall（別只看 overall accuracy）。

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=True)
import torch
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
HF_DATASET = "QCRI/MEDIC"

# ── 訓練設定 ───────────────────────────────────────────────────────
CNN_EPOCHS    = 20     # 自訓 CNN 從零訓練
CNN_LR        = 1e-3
FT_EPOCHS     = 8      # 預訓練骨幹微調
FT_LR         = 1e-4
BACKBONE      = "efficientnet_b0"   # 預訓練骨幹：可選 "efficientnet_b0" / "resnet50" / "resnet18"

BATCH_SIZE    = 64
NUM_WORKERS   = 4
SEED          = 42
MAX_SAMPLES_PER_CLASS = None  # train 每類抽樣上限（試跑可設小如 500）；None = 全量
MAX_EVAL_PER_CLASS    = None  # val/test 每類抽樣上限（試跑可設小如 200）；None = 全量
CACHE_IN_MEMORY       = True  # 影像解碼一次存記憶體；RAM 不足時設 False

# ── 5 類純災害（拿掉 MEDIC 的 not_disaster / other_disaster）──
CLASSES_EN = [
    "Earthquake Damage", "Flood", "Fire",
    "Typhoon or Storm Damage", "Landslide",
]
CLASSES_ZH = [
    "地震或建築損壞", "淹水", "火災",
    "颱風或強風災損", "土石流或坍方",
]
NUM_CLASSES = len(CLASSES_EN)  # 5

MEDIC_MAP = {
    "earthquake": 0, "flood": 1, "fire": 2,
    "hurricane": 3, "landslide": 4,
    # not_disaster / other_disaster 不列入 → MedicDataset 自動 skip
}
print("NUM_CLASSES:", NUM_CLASSES, "->", CLASSES_EN)
print("BACKBONE:", BACKBONE)


In [ ]:
import os, time, copy, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler, Dataset
from torchvision import transforms
import torchvision.models as tvm
from datasets import load_dataset
from PIL import ImageFile, Image
ImageFile.LOAD_TRUNCATED_IMAGES = True
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", message="Palette images with Transparency")
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
# （可選）HF 認證
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    print("已以 HF_TOKEN 認證")
except Exception as e:
    print(f"未使用 HF_TOKEN（{type(e).__name__}），改未認證下載")

raw = load_dataset(HF_DATASET)
print(raw)
SPLIT_TRAIN = "train"
SPLIT_DEV   = "validation" if "validation" in raw else ("dev" if "dev" in raw else None)
SPLIT_TEST  = "test" if "test" in raw else None
assert SPLIT_DEV is not None and SPLIT_TEST is not None, f"找不到 dev/test split：{list(raw.keys())}"
DT_NAMES = raw[SPLIT_TRAIN].features["disaster_types"].names
dropped = [n for n in DT_NAMES if n not in MEDIC_MAP]
print("保留 5 類災害；略過（無災害/其他）:", dropped)

In [ ]:
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

In [ ]:
class MedicDataset(Dataset):
    """把 MEDIC HF split 包成 (image_tensor, label6)；套 MEDIC_MAP（7→6）。

    每個 split 各自持有獨立 transform —— 不共享底層 dataset，
    因此不會有舊版「val transform 覆蓋 train augmentation」的副作用。
    cache=True 時於 __init__ 將影像解碼並縮到 <=256 存記憶體，
    之後每 epoch 只做 augmentation，免重複 JPEG 解碼（提高 GPU 使用率）。
    """
    def __init__(self, hf_split, transform, max_per_class=None, cache=False):
        self.hf_split  = hf_split
        self.transform = transform
        self.samples   = []                 # list of (hf_index, label6)
        self.counts    = [0] * NUM_CLASSES
        names = hf_split.features["disaster_types"].names
        for i, dt in enumerate(hf_split["disaster_types"]):
            label6 = MEDIC_MAP.get(names[dt])
            if label6 is None:
                continue
            # 注意：max_per_class 取每類在原始 dataset 的「前 N 筆」（未洗牌），純為控訓練時間；若在意分布偏差可改先洗牌再截取。
            if max_per_class is not None and self.counts[label6] >= max_per_class:
                continue
            self.counts[label6] += 1
            self.samples.append((i, label6))

        # 可選：解碼一次存記憶體（搭配 num_workers=0，避免多進程複製快取）
        self.images = None
        if cache:
            self.images = []
            for hf_i, _ in self.samples:
                im = self.hf_split[hf_i]["image"].convert("RGB")
                im.thumbnail((256, 256))   # 等比縮到 <=256 邊長，省記憶體
                self.images.append(im.copy())
            mb = sum(im.size[0] * im.size[1] * 3 for im in self.images) / 1e6
            print(f"  已快取 {len(self.images)} 張影像於記憶體（約 {mb:.0f} MB）")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        if self.images is not None:
            img = self.images[idx]
        else:
            hf_i, _ = self.samples[idx]
            img = self.hf_split[hf_i]["image"].convert("RGB")
        return self.transform(img), self.samples[idx][1]


train_ds = MedicDataset(raw[SPLIT_TRAIN], train_tf, max_per_class=MAX_SAMPLES_PER_CLASS, cache=CACHE_IN_MEMORY)
val_ds   = MedicDataset(raw[SPLIT_DEV],   val_tf,  max_per_class=MAX_EVAL_PER_CLASS, cache=CACHE_IN_MEMORY)
test_ds  = MedicDataset(raw[SPLIT_TEST],  val_tf,  max_per_class=MAX_EVAL_PER_CLASS, cache=CACHE_IN_MEMORY)

print("Train 類別分布：")
for i, (en, n) in enumerate(zip(CLASSES_EN, train_ds.counts)):
    print(f"  [{i}] {en:25s} {n:6d}")
print(f"Train: {len(train_ds)}  Dev: {len(val_ds)}  Test: {len(test_ds)}")

# sqrt-inverse 類別權重 → WeightedRandomSampler（處理嚴重不平衡）
counts   = np.array(train_ds.counts, dtype=float)
class_w  = 1.0 / np.sqrt(np.maximum(counts, 1.0))
sample_w = [class_w[label] for _, label in train_ds.samples]
sampler  = WeightedRandomSampler(sample_w, num_samples=len(sample_w), replacement=True)

# 快取在記憶體時用單進程（避免每個 worker 複製整份快取）；否則開多 worker 平行解碼
if CACHE_IN_MEMORY:
    _loader_kw = dict(num_workers=0, pin_memory=True)
else:
    _loader_kw = dict(num_workers=NUM_WORKERS, pin_memory=True)
    if NUM_WORKERS > 0:
        _loader_kw.update(persistent_workers=True, prefetch_factor=4)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, **_loader_kw)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, **_loader_kw)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, **_loader_kw)


In [ ]:
class DisasterCNN_v1(nn.Module):
    """4-block CNN baseline for 6-class disaster classification."""
    def __init__(self, num_classes: int = 6):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: 224 -> 112
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            # Block 2: 112 -> 56
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            # Block 3: 56 -> 28
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            # Block 4: 28 -> 14
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

In [ ]:
def build_backbone(name, num_classes):
    """ImageNet 預訓練骨幹，換成 num_classes 類分類頭，全層可微調。"""
    if name == "resnet18":
        net = tvm.resnet18(weights=tvm.ResNet18_Weights.IMAGENET1K_V1)
        net.fc = nn.Linear(net.fc.in_features, num_classes)
    elif name == "resnet50":
        net = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V2)
        net.fc = nn.Linear(net.fc.in_features, num_classes)
    elif name == "efficientnet_b0":
        net = tvm.efficientnet_b0(weights=tvm.EfficientNet_B0_Weights.IMAGENET1K_V1)
        net.classifier[1] = nn.Linear(net.classifier[1].in_features, num_classes)
    else:
        raise ValueError(f"未知 backbone: {name}")
    return net

def train_model(model_fn, name, epochs, lr):
    """通用訓練（AMP + cosine + label smoothing），回傳 (best_val_acc, best_state)。"""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model  = model_fn(NUM_CLASSES).to(device)
    opt    = torch.optim.Adam(model.parameters(), lr=lr)
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler = torch.amp.GradScaler("cuda") if device == "cuda" else None
    best_acc, best_state = 0.0, None
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n=== {name} ===  可訓練參數: {n_params:,}")
    for ep in range(1, epochs + 1):
        t0 = time.time()
        model.train()
        for imgs, labels in train_loader:
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            opt.zero_grad()
            if scaler is not None:
                with torch.amp.autocast("cuda"):
                    loss = loss_fn(model(imgs), labels)
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            else:
                loss = loss_fn(model(imgs), labels); loss.backward(); opt.step()
        model.eval(); correct = total = 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs = imgs.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                correct += (model(imgs).argmax(1) == labels).sum().item()
                total += labels.size(0)
        acc = correct / total
        if acc > best_acc:
            best_acc, best_state = acc, copy.deepcopy(model.state_dict())
        sched.step()
        print(f"  Epoch {ep:2d}/{epochs}  Val Acc: {acc:.4f}  ({time.time()-t0:.1f}s)")
    print(f"  → Best Val Acc: {best_acc:.4f}")
    return best_acc, best_state


@torch.no_grad()
def eval_on_test(model_fn, state):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model_fn(NUM_CLASSES).to(device); model.load_state_dict(state); model.eval()
    preds, trues = [], []
    for imgs, labels in test_loader:
        imgs = imgs.to(device, non_blocking=True)
        preds.extend(model(imgs).argmax(1).cpu().tolist())
        trues.extend(labels.tolist())
    return np.array(preds), np.array(trues)

In [ ]:
cnn_acc, cnn_state = train_model(DisasterCNN_v1, "DisasterCNN_v1 (5cls, from-scratch)", CNN_EPOCHS, CNN_LR)

In [ ]:
bk_acc, bk_state = train_model(lambda nc: build_backbone(BACKBONE, nc), f"{BACKBONE}-FT (5cls)", FT_EPOCHS, FT_LR)


In [ ]:
cnn_pred, cnn_true = eval_on_test(DisasterCNN_v1, cnn_state)
bk_pred,  bk_true  = eval_on_test(lambda nc: build_backbone(BACKBONE, nc), bk_state)

cmp = pd.DataFrame([
    {"model": "自訓 CNN (from-scratch, 5cls)", "params": sum(p.numel() for p in DisasterCNN_v1(NUM_CLASSES).parameters()), "test_acc": round(float((cnn_pred == cnn_true).mean()), 4)},
    {"model": f"{BACKBONE}-FT (5cls)",          "params": sum(p.numel() for p in build_backbone(BACKBONE, NUM_CLASSES).parameters()),  "test_acc": round(float((bk_pred == bk_true).mean()), 4)},
])
print(cmp.to_string(index=False))
print("\n=== 自訓 CNN — Test (5 類) ===")
print(classification_report(cnn_true, cnn_pred, target_names=CLASSES_EN, digits=4))
print(f"=== {BACKBONE}-FT — Test (5 類) ===")
print(classification_report(bk_true, bk_pred, target_names=CLASSES_EN, digits=4))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, (title, yt, yp) in zip(axes, [("自訓 CNN", cnn_true, cnn_pred), (f"{BACKBONE}-FT", bk_true, bk_pred)]):
    cm = confusion_matrix(yt, yp, labels=list(range(NUM_CLASSES)))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASSES_EN, yticklabels=CLASSES_EN, ax=ax)
    ax.set_title(f"{title} (Test Acc {float((yp == yt).mean()):.3f})")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.savefig("/kaggle/working/5class_cm.png", dpi=120); plt.show()


## 14. 儲存與下載權重

Run All 後，在右側 **Output** 面板或下方連結下載。​
互動 session 結束會清掉 `/kaggle/working`，記得當下就下載或 **Save Version**。


In [ ]:
import json
from IPython.display import FileLink, display

# 存兩個 5 類模型權重
torch.save(bk_state, f"/kaggle/working/{BACKBONE}_5class.pth")  # 預訓練骨帹-FT
torch.save(cnn_state, "/kaggle/working/custom_cnn_5class.pth")   # 自訓 CNN（約 1.5MB）

# 存類別對照（之後推論要用）
mapping = {
    "classes":      CLASSES_EN,
    "zh_labels":    CLASSES_ZH,
    "class_to_idx": {c: i for i, c in enumerate(CLASSES_EN)},
    "num_classes":  NUM_CLASSES,
    "dataset":      "QCRI/MEDIC disaster_types（5 類，排除 not/other）",
}
with open("/kaggle/working/classes_5class.json", "w", encoding="utf-8") as f:
    json.dump(mapping, f, ensure_ascii=False, indent=2)

print(f"已存：{BACKBONE}_5class.pth / custom_cnn_5class.pth / classes_5class.json")
for p in [f"/kaggle/working/{BACKBONE}_5class.pth",
          "/kaggle/working/custom_cnn_5class.pth",
          "/kaggle/working/classes_5class.json",
          "/kaggle/working/5class_cm.png"]:
    display(FileLink(p))
